# Load, cleanse and persist data using NumPy, Pandas and MySQL Connector/Python

In [ ]:
#!pip install numpy pandas scikit-learn mysql-connector-python

import os
from pprint import pprint

import numpy as np
import pandas as pd
import mysql.connector as sq

from sklearn.preprocessing import LabelEncoder


## Load Data from CSV into Pandas DataFrame

In [ ]:
path_to_raw_data_file = "./modified_kaggle_automobile_data.csv"

load_with_pandas = True  # Set to True to load data using Pandas, False to load using NumPy

if load_with_pandas:
    # Load the raw data from the CSV file into a Pandas DataFrame
    print("Loading raw data from CSV file into a Pandas DataFrame...")
    df = pd.read_csv(path_to_raw_data_file)
    df.info()
    df.head()
else:
    # Load the raw data from the CSV file into a NumPystructured array
    print("Loading raw data from CSV file into a NumPy structured array...")

    # genfromtxt() is more robust than loadtxt() as it can can handle CSV files with missing values, 
    # mixed data types (strings and numbers), and more complex structures.
    data_arr = np.genfromtxt(path_to_raw_data_file, delimiter=",", dtype=None, names=True, encoding="utf-8")
    print(type(data_arr))
    print()
    print(data_arr.shape)
    print()
    print(data_arr.dtype)
    print()
    print(data_arr[:5])
    print()

    # Convert the NumPy structured array to a Pandas DataFrame
    df = pd.DataFrame(data_arr)
    df.info()
    print()
    
    # Rename columns to match the original dataset's column names
    cols_to_rename = {
        "normalizedlosses": "normalized-losses",
        "fueltype": "fuel-type",
        "numofdoors": "num-of-doors",
        "bodystyle": "body-style",
        "drivewheels": "drive-wheels",
        "enginelocation": "engine-location",
        "wheelbase": "wheel-base",
        "curbweight": "curb-weight",
        "enginetype": "engine-type",
        "numofcylinders": "num-of-cylinders",
        "enginesize": "engine-size",
        "fuelsystem": "fuel-system",
        "compressionratio": "compression-ratio",
        "peakrpm": "peak-rpm",
        "citympg": "city-mpg",
        "highwaympg": "highway-mpg"
    }
    df.rename(columns=cols_to_rename, inplace=True)
    df.info()
    print()
    
    # Convert specified columns from int64 to float64
    cols_to_convert = ["symboling", "curb-weight", "city-mpg", "highway-mpg"]
    df[cols_to_convert] = df[cols_to_convert].astype("float64")

    # Replace 'NaN' string values in the 'price' column with a space character to avoid issues during conversion to numeric type
    df["price"] = pd.to_numeric(df["price"], errors='coerce')
    # df["price"] = df["price"].fillna(0)
    # df["price"] = df["price"].astype(int)

    df.info()
    df.head()


## Data Cleansing

### Col Headings

In [ ]:
# Drop irrelevant cols from the DataFrame
print("\nCol names before dropping:\n", df.columns.tolist())
print("Number of cols before dropping:", len(df.columns))

cols_to_drop = ["symboling", "normalized-losses", "num-of-doors", "body-style", "drive-wheels", "engine-location", "wheel-base", "length", "width", "height", "city-mpg", "highway-mpg"]
df.drop(columns=cols_to_drop, inplace=True)

print("\nCol names after dropping:\n", df.columns.tolist())
print("Number of cols after dropping:", len(df.columns))

# Replace dashes with underscores in col headers
df.columns = df.columns.str.replace("-", "_")
print("\nUpdated Col names:\n", df.columns.tolist())

# Rename column(s)
df.rename(columns={"horsepower": "hp", "peak_rpm": "rpm"}, inplace=True)
col_list = df.columns.tolist()
print("\nUpdated Col List:\n", col_list)


In [ ]:
print()
df.info()
df.head()


### Data Cleansing - Data

In [ ]:
# Drop completely empty rows from the DataFrame
print("Number of rows before dropping empty rows:", len(df))
df.dropna(how="all", inplace=True)
print("Number of rows after dropping empty rows:", len(df))

# Convert cells with "?" to NaN 
df.replace("?", np.nan, inplace=True)

# Convert cells with empty strings to NaN
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)

# Drop rows with NaN in any of the cols
print("\nNumber of rows before dropping rows with NaN in any col:", len(df))
df.dropna(how="any", inplace=True)
print("Number of rows after dropping rows with NaN in any col:", len(df))

# Drop rows that have "l" in the "engine_type" column
print("\nNumber of rows before dropping rows with 'l' in engine_type col:", len(df))
df.drop(df[df["engine_type"] == "l"].index, inplace=True)
print("Number of rows after dropping rows with 'l' in engine_type col:", len(df))

# Find and remove duplicate rows from the DataFrame
duplicates = df[df.duplicated()]
print("\nNumber of rows before removing duplicates:", len(df))
df.drop_duplicates(inplace=True)
print("Number of rows after removing duplicates:", len(df))


In [ ]:
df.info()
df.describe()


In [ ]:
# Change Data Types of Columns to Appropriate Types
df["curb_weight"] = df["curb_weight"].apply(np.floor).astype(int)

df["bore"] = df["bore"].astype(float)
df["stroke"] = df["stroke"].astype(float)

df["hp"] = df["hp"].astype(int)
df["rpm"] = df["rpm"].astype(int)
df["price"] = df["price"].astype(int)


In [ ]:
df.info()
print()
print(df.head())
df.describe()


### Categorical Feature Cleansing

In [ ]:
# Identify Categorical Columns in the DataFrame
categorical_cols = [col for col in df.columns if df[col].dtype == "object"]
print("\nCategorical Columns:\n", categorical_cols)

# Create a new DataFrame that contains only the categorical columns from the original DataFrame
categorical_df = df[categorical_cols]
categorical_df.head()

# Print the unique values for each categorical column in the new DataFrame
print()
for col in categorical_df.columns:
    print(f"{col}: \n{categorical_df[col].unique()}\n")


In [ ]:
# Map num_of_cylinders to numeric values in the categorical_df DataFrame
cylinder_mapping = {
    "four": 4,
    "five": 5,
    "six": 6,
    "eight": 8,
    "twelve": 12
}

# Replace the values in the num_of_cylinders column using the mapping
categorical_df.loc[:, "num_of_cylinders"] = categorical_df["num_of_cylinders"].map(cylinder_mapping)

# Change the data type of the num_of_cylinders column to int
categorical_df["num_of_cylinders"] = categorical_df["num_of_cylinders"].astype(int)

categorical_df.info()
categorical_df.head()


In [ ]:
# Label encode make, fuel_type and aspiration cols
le = LabelEncoder()
categorical_df.loc[:, "make"] = le.fit_transform(categorical_df["make"])
categorical_df.loc[:, "fuel_type"] = le.fit_transform(categorical_df["fuel_type"])
categorical_df.loc[:, "aspiration"] = le.fit_transform(categorical_df["aspiration"]) 

# Change the data type of the make, fuel_type and aspiration columns to int
categorical_df["make"] = categorical_df["make"].astype(int)
categorical_df["fuel_type"] = categorical_df["fuel_type"].astype(int)
categorical_df["aspiration"] = categorical_df["aspiration"].astype(int)

categorical_df.info()
categorical_df.head()


In [ ]:
categorical_df.tail()


In [ ]:
# One-hot encode the remaining categorical columns in the categorical_df DataFrame
one_hot_encode_cols = ["engine_type", "fuel_system"]
categorical_df = pd.get_dummies(categorical_df, columns=one_hot_encode_cols, dtype=int)

categorical_df.info()
categorical_df.head()


In [ ]:
# Check for any remaining categorical columns in the categorical_df DataFrame
categorical_df_categorical_cols = [col for col in categorical_df.columns if categorical_df[col].dtype == "object"]
print("\nCategorical Columns:\n", categorical_df_categorical_cols)

# Check for any remaining categorical columns in the orignal DataFrame
df_categorical_cols = [col for col in df.columns if df[col].dtype == "object"]
print("\nCategorical Columns:\n", df_categorical_cols)


In [ ]:
# Drop categorical columns from original DataFrame
numerical_df = df.drop(columns=df_categorical_cols)

# Check for any remaining categorical columns
numerical_df_categorical_cols = [col for col in numerical_df.columns if numerical_df[col].dtype == "object"]
print("\nCategorical Columns:\n", numerical_df_categorical_cols)

numerical_df.info()
numerical_df.head()


In [ ]:
# Combine the numerical_df and categorical_df DataFrames into a single DataFrame
data_df = pd.concat([numerical_df, categorical_df], axis=1)
data_df.info()
data_df.head()


In [ ]:
# Save DataFrame to CSV
clean_data_file = "data.csv"
data_df.to_csv(clean_data_file, index=True, header=False)


## Load Cleansed data from CSV into MySQL (MySQL Connector/Python)

In [ ]:
# Copy the cleansed data file to the MySQL data directory for loading into the database
# !rm -f /opt/homebrew/var/mysql/data.csv
!cp "./data.csv" /opt/homebrew/var/mysql/data.csv
!ls -al /opt/homebrew/var/mysql/data.csv


In [ ]:
"""
Set environment variables for MySQL connection (replace with your actual values)

These lines are for demonstration purposes only! 

In a real-world scenario, you would set these environment variables outside of your code, 
for example in your shell or CI/CD pipeline.
"""
# os.environ["MYSQL_HOST"] = "your_mysql_host"
# os.environ["MYSQL_USER"] = "your_mysql_user"
# os.environ["MYSQL_USER_PWD"] = "your_mysql_password"

In [ ]:
#Get env vars
mysql_host = os.environ.get("MYSQL_HOST")
mysql_user = os.environ.get("MYSQL_USER")
mysql_user_password = os.environ.get("MYSQL_USER_PWD")

#Establish connection to DB
try:
    mydb=sq.connect(host=mysql_host,user=mysql_user,passwd=mysql_user_password, buffered=True)

    if mydb.is_connected():
        print("MySQL connection established successfully.")

        # Create Cursor - this will be used to execute SQL queries against the database
        mycursor = mydb.cursor()

        # Check if database exists
        mycursor.execute("SHOW DATABASES LIKE 'cleansed_automobobile_data'")
        db_exists = mycursor.fetchone()

        if not db_exists:
            print("Database 'cleansed_automobobile_data' does not exist.\nCreating database...")
            mycursor.execute("CREATE DATABASE cleansed_automobobile_data")
            print("Database 'cleansed_automobobile_data' created successfully.")

            # Switch to the 'cleansed_automobobile_data' database
            mycursor.execute("USE cleansed_automobobile_data")

            # Create the 'automobile_data' table in the 'cleansed_automobobile_data' database
            sql_str = """
            CREATE TABLE cleansed_automobobile_data.automobile_data (
                id INT NOT NULL,
                curb_weight INT NOT NULL,
                engine_size FLOAT NOT NULL,
                bore FLOAT NOT NULL,
                stroke FLOAT NOT NULL,
                compression_ratio FLOAT NOT NULL,
                hp INT NOT NULL,
                rpm INT NOT NULL,
                price INT NOT NULL,
                make INT NOT NULL,
                fuel_type INT NOT NULL,
                aspiration INT NOT NULL,
                num_of_cylinders INT NOT NULL,
                engine_type_dohc INT NOT NULL,
                engine_type_ohc INT NOT NULL,
                engine_type_ohcf INT NOT NULL,
                engine_type_ohcv INT NOT NULL,
                fuel_system_1bbl INT NOT NULL,
                fuel_system_2bbl INT NOT NULL,
                fuel_system_idi INT NOT NULL,
                fuel_system_mfi INT NOT NULL,
                fuel_system_mpfi INT NOT NULL,
                fuel_system_spdi INT NOT NULL,
                fuel_system_spfi INT NOT NULL,
                PRIMARY KEY(id)
            );
            """
            mycursor.execute(sql_str)

            # Describe the newly created table
            mycursor.execute("DESCRIBE automobile_data;")
            table_description = mycursor.fetchall()
            print("\nTable 'automobile_data' structure:")
            pprint(table_description)

            # Insert data from the CSV file into the newly created table using LOAD DATA INFILE
            sql_str = "LOAD DATA INFILE './data.csv' \
            INTO TABLE cleansed_automobobile_data.automobile_data \
            FIELDS TERMINATED BY ','  OPTIONALLY ENCLOSED BY '\"' LINES TERMINATED BY '\\n' IGNORE 1 ROWS;"
            mycursor.execute(sql_str)
            mydb.commit()

            sql_str = "select count(*) from cleansed_automobobile_data.automobile_data;"
            mycursor.execute(sql_str)
            count = mycursor.fetchone()[0]

            print(f"\nNumber of rows in cleansed_automobobile_data.automobile_data table: {count}")
        else:
            print("Database 'cleansed_automobobile_data' already exists.")

        mycursor.close()
        print("MySQL cursor closed successfully.")
    else:
        print("Failed to establish MySQL connection.")
except Exception as e:
    print(f"Error connecting to MySQL: {e}")


## Load data from DB into DataFrame

In [ ]:
data_for_model_df = pd.read_sql("SELECT * FROM cleansed_automobobile_data.automobile_data", con=mydb)
data_for_model_df.drop(columns=["id"], inplace=True)

data_for_model_df.info()
print()
print(data_for_model_df.head())
data_for_model_df.describe()
